In [2]:
import torch
import pandas as pd 

In [3]:
data=pd.read_csv('../../data/100_Unique_QA_Dataset.csv')

In [4]:
data

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [ ]:
#tokenize

def tokenize(text):
    text=text.lower()
    text=text.replace('?','')
    text=text.replace("'","")
    
    return text.split()

In [8]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [19]:
#vocab
vocab={'<UNK>':0}

def build_vocab(row):
    token_question=tokenize(row['question'])
    token_answer=tokenize(row['answer'])
    
    merged_tokens=token_question + token_answer
    
    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)


In [20]:
data.apply(build_vocab,axis=1) #axis=1 means whole row 

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [21]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [22]:
print(len(vocab))

324


In [29]:
def text_to_indexs(text,vocab):
    indexed_text=[]
    
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
        
    return indexed_text

In [30]:
text_to_indexs('my name is suyog karki ',vocab)

[0, 0, 2, 0, 0]

In [98]:
len(vocab)

324

In [31]:
from torch.utils.data import Dataset,DataLoader

In [ ]:



class QADataset(Dataset):
    def __init__(self,data,vocab):
        self.data=data
        self.vocab=vocab
        
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, index):
        numerical_question=text_to_indexs(self.data.iloc[index]['question'],self.vocab)
        numerical_answer=text_to_indexs(self.data.iloc[index]['answer'],self.vocab)
        
        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [38]:
dataset=QADataset(data,vocab)

In [39]:
dataset[2]

(tensor([10, 11, 12, 13, 14, 15]), tensor([16]))

In [40]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [49]:
for question , answer in dataloader:
    print(question,answer[0])

tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([254])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([287])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([298])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([136])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([280])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([61])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([276])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]) tensor([244])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([205])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([99])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([85])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([238])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([170])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([188])
tensor([[ 42, 10

In [42]:
import torch.nn as nn

In [ ]:



class SImpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn=nn.RNN(50,64,batch_first=True)
        self.fc=nn.Linear(64,vocab_size)
        
    def forward(self,question):
        embedded_question=self.embedding(question)
        hidden,final=self.rnn(embedded_question)
        output=self.fc(final.squeeze(0))    
        
        return output

In [58]:
learning_rate=0.001
epochs=20

In [59]:
model=SImpleRNN(len(vocab))

In [60]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [61]:
#training loop

for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        output=model(question)
        loss=criterion(output,answer[0])
        
        loss.backward()
        
        optimizer.step()
        
        total_loss=total_loss+loss.item()
    print(f'Epoch {epoch +1},loss:{total_loss:4f}')

Epoch 1,loss:528.343049
Epoch 2,loss:457.512555
Epoch 3,loss:377.533732
Epoch 4,loss:317.140823
Epoch 5,loss:265.328580
Epoch 6,loss:215.832136
Epoch 7,loss:171.633392
Epoch 8,loss:133.105733
Epoch 9,loss:102.052565
Epoch 10,loss:77.974262
Epoch 11,loss:60.020506
Epoch 12,loss:47.405861
Epoch 13,loss:37.533688
Epoch 14,loss:30.488383
Epoch 15,loss:25.611610
Epoch 16,loss:21.435977
Epoch 17,loss:18.314398
Epoch 18,loss:15.712827
Epoch 19,loss:13.695085
Epoch 20,loss:11.925149


In [91]:
def predict(model,question,threshold=0.5):
    
    #covert question to numbers
    numerical_question=text_to_indexs(question,vocab)
    
    #tensor
    question_tensor=torch.tensor(numerical_question).unsqueeze(0)
    
    #send to model
    output=model(question_tensor)
    
    #conv=ert logits to proba 
    proba=torch.nn.functional.softmax(output,dim=1)
    
    #find max proba 
    value, index= torch.max(proba,dim=1)
    
    if value < threshold:
        print('idk')
    else:
        
        print(list(vocab.keys())[index])

In [97]:
predict(model,"Great Wall")

china
